# FRED - Stagflation Era International Comparison (1970-1995)

Three indicators per country - quarterly real GDP growth, year-on-year CPI inflation, and the unemployment rate - plotted with goldenrod shading over runs of 2+ consecutive quarters of negative real GDP growth (the technical-recession definition).

**Sources**: FRED for most series; DBnomics-served OECD Quarterly National Accounts for Japan, New Zealand, and Italy quarterly real GDP (FRED's own mirrors for these start too late); OECD Economic Outlook annual UR (interpolated) back-filled pre-1983 for France and Italy.

**Coverage caveats** (also shown as per-chart `rheader` notes where relevant):
- **West Germany**: shown to 1991Q2 only; FRED's German real-GDP series is West-then-Unified spliced.
- **France**: UR is a splice - FRED/OECD harmonised quarterly UR (1983Q1+) concatenated with OECD Economic Outlook annual UR (interpolated to quarterly) pre-1983. Expect a small level/shape discontinuity at the 1982/1983 boundary.
- **Italy**: real GDP now backfilled to 1960Q1 via OECD QNA (DBnomics). UR spliced the same way as France (quarterly FRED from 1983Q1+, OECD EO annual interpolated pre-1983).
- **New Zealand**: unemployment rate from 1986Q1 only (Stats NZ HLFS begins then). GDP and CPI go back to 1960.
- **Japan**: GDP, CPI and UR all cover the full window. The 2+-consecutive-negatives rule only fires in 1993Q2-Q3 for Japan; the 1974 oil-shock contraction shows as volatile single-quarter dips in the annualised SA series and doesn't satisfy the rule.

Data sources: https://fred.stlouisfed.org/ and https://db.nomics.world/

## Setup

In [1]:
# system imports
from pathlib import Path
import time

# analytic imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

# local imports
import mgplot as mg

In [2]:
# pandas display
pd.options.display.max_rows = 999999

# Display charts inline?
SHOW = False

# Window for analysis - fixed 1970-1995 x-axis applied to every panel in every
# figure so all countries are directly comparable.
WINDOW_START = pd.Period("1970Q1", freq="Q")
WINDOW_END = pd.Period("1995Q4", freq="Q")

# Footer source attribution
SOURCE = "FRED + DBnomics (OECD QNA for Japan/NZ/Italy GDP)"
RECESSION_NOTE = (
    "Goldenrod shading: 2+ consecutive quarters of negative real Q/Q GDP growth. "
)

# Chart directory
CHART_DIR = "./CHARTS/Stagflation/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

In [3]:
# Load FRED API key
FRED_API_KEY = Path("fred.api").read_text().strip()
FRED_OBS_URL = "https://api.stlouisfed.org/fred/series/observations"

## Country / series specification

In [4]:
# For each country: real-GDP series (level/index), CPI series (index), UR series (%),
# an optional `end` Period to truncate (used for West Germany pre-unification), and
# an optional `note` string shown as rheader on that country's figure.
# Source tag: "fred:SID" or "dbnomics:PROVIDER/DATASET/SERIES".
# A list of source tags triggers splicing: the first tag is preferred; later
# tags fill gaps (used for France/Italy UR, where the quarterly series begins
# 1983Q1 and we back-fill from an annual series).
# All GDP series are SA; CPI series are NSA (YoY removes seasonality); UR series are SA.
COUNTRIES: dict[str, dict] = {
    "USA":          {"gdp": "fred:GDPC1",                            "cpi": "fred:CPIAUCSL",        "ur": "fred:UNRATE",                             "end": None,                                 "note": None},
    "UK":           {"gdp": "fred:NAEXKP01GBQ661S",                  "cpi": "fred:GBRCPIALLMINMEI", "ur": "fred:LRUNTTTTGBQ156S",                    "end": None,                                 "note": None},
    "Canada":       {"gdp": "fred:NGDPRSAXDCCAQ",                    "cpi": "fred:CANCPIALLMINMEI", "ur": "fred:LRHUTTTTCAQ156S",                    "end": None,                                 "note": None},
    "Australia":    {"gdp": "fred:NGDPRSAXDCAUQ",                    "cpi": "fred:AUSCPIALLQINMEI", "ur": "fred:LRHUTTTTAUQ156S",                    "end": None,                                 "note": None},
    "New Zealand":  {"gdp": "dbnomics:OECD/QNA/NZL.B1_GE.VOBARSA.Q", "cpi": "fred:NZLCPIALLQINMEI", "ur": "dbnomics:OECD/MEI/NZL.LRUNTTTT.STSA.A",   "end": None,                                 "note": "NZ UR is an annual rate (interpolated to quarterly); GDP pre-1987 is interpolated from annual data."},
    "France":       {"gdp": "fred:NAEXKP01FRQ661S",                  "cpi": "fred:FRACPIALLMINMEI", "ur": ["fred:LRHUTTTTFRQ156S", "dbnomics:OECD/EO/FRA.UNR.A"], "end": None,                    "note": "UR: FRED/OECD harmonised quarterly series (1983Q1+) spliced with OECD Economic Outlook annual UR (interpolated to quarterly) for 1970-1982."},
    "West Germany": {"gdp": "fred:NAEXKP01DEQ661S",                  "cpi": "fred:DEUCPIALLMINMEI", "ur": "fred:LRUNTTTTDEQ156S",                    "end": pd.Period("1991Q2", freq="Q"),        "note": "Truncated at 1991Q2: FRED's German real-GDP series switches from West Germany to Unified Germany after reunification."},
    "Italy":        {"gdp": "dbnomics:OECD/QNA/ITA.B1_GE.VOBARSA.Q", "cpi": "fred:ITACPIALLMINMEI", "ur": ["fred:LRHUTTTTITQ156S", "dbnomics:OECD/EO/ITA.UNR.A"], "end": None,                    "note": "UR: FRED/OECD harmonised quarterly series (1983Q1+) spliced with OECD Economic Outlook annual UR (interpolated to quarterly) for 1970-1982."},
    "Japan":        {"gdp": "dbnomics:OECD/QNA/JPN.B1_GE.VOBARSA.Q", "cpi": "fred:JPNCPIALLMINMEI", "ur": "fred:LRHUTTTTJPQ156S",                    "end": None,                                 "note": None},
}

## FRED fetch helpers

In [5]:
def fetch_fred(series_id: str, start: str = "1965-01-01", retries: int = 4) -> pd.Series:
    """Fetch a FRED series as a pandas Series with DatetimeIndex.
    Retries with linear backoff on transient 5xx errors."""
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start,
    }
    for attempt in range(retries):
        r = requests.get(FRED_OBS_URL, params=params, timeout=30)
        if r.status_code == 200:
            break
        if r.status_code >= 500 and attempt < retries - 1:
            time.sleep(1.0 * (attempt + 1))
            continue
        r.raise_for_status()
    obs = r.json()["observations"]
    dates = pd.to_datetime([o["date"] for o in obs])
    vals = [float(o["value"]) if o["value"] != "." else np.nan for o in obs]
    return pd.Series(vals, index=dates, name=series_id).dropna()


DBNOMICS_BASE = "https://api.db.nomics.world/v22"


def _to_float(v) -> float:
    """Coerce DBnomics observation value to float; 'NA'/None/'' -> NaN."""
    if v is None:
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str) and v.strip() in ("", "NA", "na", "."):
        return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def fetch_dbnomics(path: str, retries: int = 4) -> pd.Series:
    """Fetch a DBnomics series by 'PROVIDER/DATASET/SERIES' path.
    Returns a pandas Series with PeriodIndex; frequency (A/Q) auto-detected
    from the first period string. Retries on timeout or 5xx."""
    last_exc: Exception | None = None
    for attempt in range(retries):
        try:
            r = requests.get(
                f"{DBNOMICS_BASE}/series/{path}",
                params={"observations": "1"},
                timeout=60,
            )
            if r.status_code >= 500 and attempt < retries - 1:
                time.sleep(1.0 * (attempt + 1))
                continue
            r.raise_for_status()
            break
        except (requests.Timeout, requests.ConnectionError) as exc:
            last_exc = exc
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            raise
    docs = r.json().get("series", {}).get("docs", [])
    if not docs:
        raise ValueError(f"No DBnomics data for {path}")
    doc = docs[0]
    periods = doc.get("period", [])
    values = doc.get("value", [])
    if not periods:
        raise ValueError(f"Empty series: {path}")
    first = pd.Period(periods[0])
    freq = first.freqstr
    idx = pd.PeriodIndex([pd.Period(p, freq=freq) for p in periods])
    vals = [_to_float(v) for v in values]
    return pd.Series(vals, index=idx, dtype="float64", name=path).dropna()


def fetch_series(tagged: str) -> pd.Series:
    """Dispatch on 'fred:...' / 'dbnomics:...' source tag."""
    if tagged.startswith("fred:"):
        return fetch_fred(tagged.removeprefix("fred:"))
    if tagged.startswith("dbnomics:"):
        return fetch_dbnomics(tagged.removeprefix("dbnomics:"))
    raise ValueError(f"Unknown source tag in {tagged!r}")

In [6]:
def to_quarterly(s: pd.Series) -> pd.Series:
    """Convert any supported index to a quarterly PeriodIndex.
    - Quarterly PeriodIndex: pass-through.
    - Annual PeriodIndex: place each annual value at Q2 (mid-year), then
      reindex to full quarterly range and linearly interpolate.
    - DatetimeIndex (monthly or finer): resample to quarter mean."""
    if isinstance(s.index, pd.PeriodIndex):
        freq = s.index.freqstr
        if freq.startswith("Q"):
            return s.copy()
        if freq.startswith("A") or freq.startswith("Y"):
            mid = pd.PeriodIndex(
                [pd.Period(f"{p.year}Q2", freq="Q") for p in s.index],
                freq="Q",
            )
            anchored = pd.Series(s.values, index=mid, name=s.name)
            full = pd.period_range(
                anchored.index.min(), anchored.index.max() + 2, freq="Q"
            )
            return anchored.reindex(full).interpolate(method="linear")
        raise ValueError(f"Unsupported PeriodIndex freq: {freq}")
    out = s.resample("QE").mean()
    out.index = pd.PeriodIndex(out.index, freq="Q")
    return out.dropna()


def splice_series(parts: list[pd.Series]) -> pd.Series:
    """Splice multiple quarterly series, preferring earlier entries.
    The first series wins wherever it has data; later series fill gaps
    (typically earlier periods). All inputs must already be quarterly."""
    if not parts:
        raise ValueError("splice_series requires at least one series")
    if len(parts) == 1:
        return parts[0].copy()
    full = pd.period_range(
        min(p.index.min() for p in parts),
        max(p.index.max() for p in parts),
        freq="Q",
    )
    merged = parts[0].reindex(full)
    for p in parts[1:]:
        merged = merged.fillna(p.reindex(full))
    return merged.dropna()


def resolve_source(spec_value: str | list[str]) -> pd.Series:
    """Fetch and quarterise one source tag, or splice a list of them."""
    if isinstance(spec_value, str):
        return to_quarterly(fetch_series(spec_value))
    parts = [to_quarterly(fetch_series(t)) for t in spec_value]
    return splice_series(parts)

## Recession-span identification

In [7]:
def find_recession_spans(
    gdp_qoq: pd.Series, min_run: int = 2
) -> list[tuple[pd.Period, pd.Period]]:
    """Return list of (start, end-inclusive) Period tuples for runs of
    `min_run`+ consecutive negative observations in `gdp_qoq`."""
    g = gdp_qoq.dropna()
    is_neg = (g < 0).to_numpy()
    idx = g.index
    spans, i, n = [], 0, len(is_neg)
    while i < n:
        if is_neg[i]:
            j = i
            while j + 1 < n and is_neg[j + 1]:
                j += 1
            if j - i + 1 >= min_run:
                spans.append((idx[i], idx[j]))
            i = j + 1
        else:
            i += 1
    return spans

## Fetch all data

In [8]:
def fetch_all() -> dict[str, dict[str, pd.Series | None]]:
    """Fetch every series for every country and convert to quarterly PeriodIndex."""
    out: dict[str, dict[str, pd.Series | None]] = {}
    for country, spec in COUNTRIES.items():
        out[country] = {}
        for measure in ("gdp", "cpi", "ur"):
            tagged = spec[measure]
            if tagged is None:
                out[country][measure] = None
                continue
            try:
                out[country][measure] = resolve_source(tagged)
            except Exception as exc:
                print(f"  failed {country} {measure}={tagged}: {exc}")
                out[country][measure] = None
            time.sleep(0.05)
        print(
            f"{country:14s}  GDP={'-' if out[country]['gdp'] is None else len(out[country]['gdp']):>4}  "
            f"CPI={'-' if out[country]['cpi'] is None else len(out[country]['cpi']):>4}  "
            f"UR={'-' if out[country]['ur'] is None else len(out[country]['ur']):>4}"
        )
    return out


raw = fetch_all()

USA             GDP= 244  CPI= 245  UR= 245


UK              GDP= 235  CPI= 241  UR= 220


Canada          GDP= 244  CPI= 241  UR= 245


Australia       GDP= 244  CPI= 241  UR= 238


New Zealand     GDP= 255  CPI= 241  UR= 259


France          GDP= 235  CPI= 241  UR= 263


West Germany    GDP= 215  CPI= 241  UR= 243


Italy           GDP= 256  CPI= 241  UR= 263


Japan           GDP= 255  CPI= 226  UR= 244


## Derive Q/Q GDP growth and YoY CPI inflation

In [9]:
def derive(raw_data: dict) -> dict[str, dict[str, pd.Series | None]]:
    """Add `gdp_qoq` and `cpi_yoy` keys derived from levels."""
    derived: dict[str, dict[str, pd.Series | None]] = {}
    for country, m in raw_data.items():
        gdp_qoq = m["gdp"].pct_change() * 100 if m["gdp"] is not None else None
        cpi_yoy = m["cpi"].pct_change(4) * 100 if m["cpi"] is not None else None
        derived[country] = {
            "gdp_qoq": gdp_qoq,
            "cpi_yoy": cpi_yoy,
            "ur": m["ur"],
        }
    return derived


panel = derive(raw)

## Plotting

In [10]:
def truncate(
    s: pd.Series | None,
    start: pd.Period,
    end: pd.Period,
    end_period: pd.Period | None = None,
) -> pd.Series | None:
    """Slice a series to [start, min(end, end_period)] and drop NaN."""
    if s is None:
        return None
    upper = end if end_period is None else min(end, end_period)
    out = s.dropna()
    out = out[(out.index >= start) & (out.index <= upper)]
    return out if not out.empty else None


def window_align(s: pd.Series | None) -> pd.Series | None:
    """Reindex a series to the full WINDOW_START..WINDOW_END quarterly range,
    padding with NaN. Ensures every panel's x-axis spans the full window
    regardless of where the underlying data starts/ends."""
    if s is None:
        return None
    full = pd.period_range(WINDOW_START, WINDOW_END, freq="Q")
    return s.reindex(full)


def recession_axvspan(
    spans: list[tuple[pd.Period, pd.Period]],
) -> list[dict] | None:
    """Convert recession spans to a list of axvspan kwargs for mgplot.finalise_plot.
    xmax extended by one quarter so the shading visually covers the last neg quarter."""
    if not spans:
        return None
    return [
        {"xmin": s, "xmax": e + 1, "color": "goldenrod", "alpha": 0.30, "zorder": -1}
        for s, e in spans
    ]

In [11]:
PANEL_HEIGHT_IN = 2.2  # inches per panel (try 3.3 for taller, 1.65 for half)


def plot_country(country: str) -> list[tuple[pd.Period, pd.Period]]:
    """Produce one figure per country with vertically stacked panels:
    real GDP Q/Q growth, CPI YoY inflation, unemployment rate.
    Goldenrod shading over 2+-quarter real-GDP contractions, applied as
    backplane to every panel. Every figure uses the same fixed x-axis
    (WINDOW_START..WINDOW_END) via reindexing. Optional per-country
    `note` is shown as rheader."""

    spec = COUNTRIES[country]
    end_period = spec["end"]
    cap_suffix = "" if end_period is None else f" (to {end_period})"

    gdp_qoq = truncate(panel[country]["gdp_qoq"], WINDOW_START, WINDOW_END, end_period)
    cpi_yoy = truncate(panel[country]["cpi_yoy"], WINDOW_START, WINDOW_END, end_period)
    ur = truncate(panel[country]["ur"], WINDOW_START, WINDOW_END, end_period)

    spans = find_recession_spans(gdp_qoq) if gdp_qoq is not None else []
    axv = recession_axvspan(spans)

    # Reindex each series to the full window so every panel auto-scales
    # to the same 1970Q1..1995Q4 x-axis.
    gdp_qoq = window_align(gdp_qoq)
    cpi_yoy = window_align(cpi_yoy)
    ur = window_align(ur)

    panel_specs: list[tuple[pd.Series, str, str]] = []
    if gdp_qoq is not None:
        panel_specs.append((gdp_qoq, "navy", "Real GDP, Q/Q growth"))
    if cpi_yoy is not None:
        panel_specs.append((cpi_yoy, "darkred", "CPI inflation, year-on-year"))
    if ur is not None:
        panel_specs.append((ur, "darkgreen", "Unemployment rate"))

    n = len(panel_specs)
    fig_size = (9.0, float(round(PANEL_HEIGHT_IN * n, 1)))
    fig, axes_obj = plt.subplots(n, 1, figsize=fig_size, sharex=True)
    axes_list = [axes_obj] if n == 1 else list(axes_obj)

    for i, (ax, (data, color, ptitle)) in enumerate(zip(axes_list, panel_specs)):
        mg.line_plot(data, ax=ax, color=color, width=1.5, annotate=False)
        kw: dict = {
            "title": ptitle,
            "ylabel": "Per cent",
            "y0": True,
        }
        if axv is not None:
            kw["axvspan"] = axv

        is_last = i == n - 1
        if is_last:
            kw.update({
                "suptitle": f"{country}: Stagflation era and its aftermath 1970-1995{cap_suffix}",
                "rfooter": SOURCE,
                "lfooter": RECESSION_NOTE,
                "figsize": fig_size,
                "tag": country,
                "show": SHOW,
            })
            if spec.get("note"):
                kw["rheader"] = spec["note"]
        else:
            kw["axes_only"] = True

        mg.finalise_plot(ax, **kw)

    return spans

## Per-country plots

In [12]:
all_spans: dict[str, list[tuple[pd.Period, pd.Period]]] = {}
for country in COUNTRIES:
    print(f"\n=== {country} ===")
    spans = plot_country(country)
    all_spans[country] = spans
    if spans:
        print(f"  Recession spans (2+ quarters of negative GDP):")
        for s, e in spans:
            print(f"    {s}  to  {e}  ({(e.ordinal - s.ordinal) + 1} quarters)")
    elif panel[country]["gdp_qoq"] is None:
        print("  GDP not available - no recession analysis.")
    else:
        print("  No 2+ negative quarters in window.")


=== USA ===


  Recession spans (2+ quarters of negative GDP):
    1974Q3  to  1975Q1  (3 quarters)
    1980Q2  to  1980Q3  (2 quarters)
    1981Q4  to  1982Q1  (2 quarters)
    1990Q4  to  1991Q1  (2 quarters)

=== UK ===


  Recession spans (2+ quarters of negative GDP):
    1973Q3  to  1974Q1  (3 quarters)
    1975Q2  to  1975Q3  (2 quarters)
    1980Q1  to  1981Q1  (5 quarters)
    1990Q3  to  1991Q3  (5 quarters)
    1992Q1  to  1992Q2  (2 quarters)

=== Canada ===


  Recession spans (2+ quarters of negative GDP):
    1974Q4  to  1975Q1  (2 quarters)
    1980Q2  to  1980Q3  (2 quarters)
    1981Q3  to  1982Q4  (6 quarters)
    1990Q2  to  1991Q1  (4 quarters)

=== Australia ===


  Recession spans (2+ quarters of negative GDP):
    1971Q4  to  1972Q1  (2 quarters)
    1975Q3  to  1975Q4  (2 quarters)
    1977Q3  to  1977Q4  (2 quarters)
    1981Q4  to  1982Q1  (2 quarters)
    1982Q3  to  1983Q2  (4 quarters)
    1991Q1  to  1991Q2  (2 quarters)

=== New Zealand ===


  Recession spans (2+ quarters of negative GDP):
    1970Q1  to  1970Q2  (2 quarters)
    1974Q4  to  1975Q3  (4 quarters)
    1976Q3  to  1977Q4  (6 quarters)
    1978Q4  to  1979Q2  (3 quarters)
    1985Q2  to  1985Q3  (2 quarters)
    1991Q1  to  1991Q2  (2 quarters)

=== France ===


  Recession spans (2+ quarters of negative GDP):
    1974Q4  to  1975Q1  (2 quarters)
    1992Q3  to  1993Q2  (4 quarters)

=== West Germany ===


  Recession spans (2+ quarters of negative GDP):
    1974Q4  to  1975Q2  (3 quarters)
    1980Q2  to  1980Q4  (3 quarters)
    1982Q2  to  1982Q3  (2 quarters)

=== Italy ===


  Recession spans (2+ quarters of negative GDP):
    1974Q4  to  1975Q2  (3 quarters)
    1977Q2  to  1977Q3  (2 quarters)
    1982Q1  to  1982Q4  (4 quarters)
    1992Q2  to  1993Q3  (6 quarters)

=== Japan ===


  Recession spans (2+ quarters of negative GDP):
    1993Q2  to  1993Q3  (2 quarters)


## Recession-span summary table

In [13]:
rows = []
for country, spans in all_spans.items():
    for s, e in spans:
        rows.append({
            "Country": country,
            "Start": str(s),
            "End": str(e),
            "Quarters": (e.ordinal - s.ordinal) + 1,
        })
summary = pd.DataFrame(rows).sort_values(["Start", "Country"]).reset_index(drop=True)
display(summary)

,Country,Start,End,Quarters
0,New Zealand,1970Q1,1970Q2,2
1,Australia,1971Q4,1972Q1,2
2,UK,1973Q3,1974Q1,3
3,USA,1974Q3,1975Q1,3
4,Canada,1974Q4,1975Q1,2
5,France,1974Q4,1975Q1,2
6,Italy,1974Q4,1975Q2,3
7,New Zealand,1974Q4,1975Q3,4
8,West Germany,1974Q4,1975Q2,3
9,UK,1975Q2,1975Q3,2


## Finished

In [14]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-24 18:21:28

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

IPython   : 9.12.0
matplotlib: 3.10.8
mgplot    : 0.2.23
numpy     : 2.4.4
pandas    : 3.0.2
pathlib   : 1.0.1
requests  : 2.33.1

Watermark: 2.6.0

